# 📊 RAG Evaluation Datasets & Benchmarks

## Why Benchmarks Matter

You can't improve what you can't measure.

Standard benchmarks let you:
- Compare your system against published research
- Track improvements as you add techniques
- Identify weaknesses (e.g., bad at multi-hop questions)

## The Most Important Benchmarks

| Dataset | What It Tests | Size |
|---|---|---|
| **MS MARCO** | Passage retrieval from Bing queries | 8.8M docs |
| **BEIR** | 18 retrieval tasks in one benchmark | Varies |
| **HotpotQA** | Multi-hop question answering | 113K Q&A |
| **NaturalQuestions** | Real Google search queries | 300K Q&A |
| **TriviaQA** | Trivia questions with evidence | 95K Q&A |
| **RAGAS Benchmark** | End-to-end RAG quality | Custom |

In [1]:
# Let's simulate a mini evaluation dataset
# and show how to compute standard retrieval metrics

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Small corpus (like a subset of MS MARCO)
corpus = [
    "Photosynthesis converts sunlight into chemical energy stored as glucose.",
    "The mitochondria produces ATP through cellular respiration.",
    "DNA replication ensures each daughter cell gets a complete genome.",
    "Enzymes are proteins that catalyse biochemical reactions.",
    "Osmosis is the movement of water across a semi-permeable membrane.",
    "The cell membrane controls what enters and exits the cell.",
    "Ribosomes are organelles where protein synthesis occurs.",
    "Chlorophyll is the pigment that makes plants appear green.",
]

corpus_embeddings = model.encode(corpus)

# Evaluation set: question → relevant doc index(es)
eval_set = [
    {"query": "How do plants make energy from sunlight?",    "relevant": [0, 7]},
    {"query": "Where does protein synthesis happen?",         "relevant": [6]},
    {"query": "What speeds up chemical reactions in cells?", "relevant": [3]},
    {"query": "How do cells produce ATP?",                   "relevant": [1]},
]

print(f"Corpus: {len(corpus)} documents")
print(f"Eval set: {len(eval_set)} questions")

Corpus: 8 documents
Eval set: 4 questions


In [2]:
# Key metric: Recall@K
# "Of the relevant docs, how many did we find in the top K results?"

def recall_at_k(retrieved_ids, relevant_ids, k):
    top_k_retrieved = set(retrieved_ids[:k])
    relevant        = set(relevant_ids)
    return len(top_k_retrieved & relevant) / len(relevant)

# Another key metric: MRR (Mean Reciprocal Rank)
# "On average, how high does the FIRST relevant doc rank?"

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0

print("Metrics defined: Recall@K and MRR")

Metrics defined: Recall@K and MRR


In [3]:
# Run evaluation on the full eval set

recall1_scores, recall3_scores, mrr_scores = [], [], []

print(f"{'Query':<45} {'Recall@1':>9} {'Recall@3':>9} {'RR':>6}")
print("-" * 73)

for item in eval_set:
    q_emb   = model.encode(item["query"])
    scores  = cosine_similarity([q_emb], corpus_embeddings)[0]
    ranked  = list(np.argsort(scores)[::-1])

    r1  = recall_at_k(ranked, item["relevant"], k=1)
    r3  = recall_at_k(ranked, item["relevant"], k=3)
    rr  = reciprocal_rank(ranked, item["relevant"])

    recall1_scores.append(r1)
    recall3_scores.append(r3)
    mrr_scores.append(rr)

    print(f"{item['query']:<45} {r1:>9.2f} {r3:>9.2f} {rr:>6.2f}")

print("-" * 73)
print(f"{'AVERAGE':<45} {np.mean(recall1_scores):>9.2f} {np.mean(recall3_scores):>9.2f} {np.mean(mrr_scores):>6.2f}")

Query                                          Recall@1  Recall@3     RR
-------------------------------------------------------------------------
How do plants make energy from sunlight?           0.50      1.00   1.00
Where does protein synthesis happen?               1.00      1.00   1.00
What speeds up chemical reactions in cells?        1.00      1.00   1.00
How do cells produce ATP?                          1.00      1.00   1.00
-------------------------------------------------------------------------
AVERAGE                                            0.88      1.00   1.00


In [4]:
# How to load real benchmarks

how_to_load = """
# MS MARCO (passage retrieval)
from datasets import load_dataset
msmarco = load_dataset("ms_marco", "v2.1")

# NaturalQuestions
nq = load_dataset("natural_questions")

# HotpotQA (multi-hop)
hotpot = load_dataset("hotpot_qa", "fullwiki")

# BEIR — the gold standard RAG benchmark suite
# pip install beir
from beir import util
from beir.datasets.data_loader import GenericDataLoader

url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip"
data_path = util.download_and_unzip(url, "datasets")
corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")
"""

print("How to load real benchmarks:")
print(how_to_load)

How to load real benchmarks:

# MS MARCO (passage retrieval)
from datasets import load_dataset
msmarco = load_dataset("ms_marco", "v2.1")

# NaturalQuestions
nq = load_dataset("natural_questions")

# HotpotQA (multi-hop)
hotpot = load_dataset("hotpot_qa", "fullwiki")

# BEIR — the gold standard RAG benchmark suite
# pip install beir
from beir import util
from beir.datasets.data_loader import GenericDataLoader

url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip"
data_path = util.download_and_unzip(url, "datasets")
corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")



## Which Benchmark to Use?

```
Building a search engine?       →  MS MARCO, BEIR
Building a Q&A bot?             →  NaturalQuestions, TriviaQA
Multi-step reasoning?           →  HotpotQA, MuSiQue
Evaluating RAG end-to-end?      →  RAGAS (faithfulness + relevance)
All-round retrieval comparison? →  BEIR (18 tasks in one)
```

**Start with BEIR** — it's the standard way to publish retrieval results and covers the most ground.